# 03 - Player segmentation (module C)

**Question.** Are there distinct player profiles in the behaviour data?

**Plan.**
1. Look at the variables (distributions, correlations).
2. Run K-means and check whether the structure is real: compare with data whose columns are shuffled
   (no structure by construction), and check stability across random seeds.
3. Keep a small number of segments, name them, and state the limits.

In [1]:
import duckdb
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.preprocessing import StandardScaler

con = duckdb.connect("../game.duckdb", read_only=True)
df = con.sql("SELECT * FROM gaming_behavior").df()
con.close()

all_vars = ["PlayTimeHours", "SessionsPerWeek", "AvgSessionDurationMinutes",
            "PlayerLevel", "AchievementsUnlocked", "InGamePurchases"]

print(df[all_vars].describe().round(2))
print(df[all_vars].corr().round(2))          # near-zero correlations = variables almost independent
print(df["InGamePurchases"].value_counts())  # binary: 0 = never paid, 1 = paid

       PlayTimeHours  SessionsPerWeek  AvgSessionDurationMinutes  PlayerLevel  \
count       40034.00         40034.00                   40034.00     40034.00   
mean           12.02             9.47                      94.79        49.66   
std             6.91             5.76                      49.01        28.59   
min             0.00             0.00                      10.00         1.00   
25%             6.07             4.00                      52.00        25.00   
50%            12.01             9.00                      95.00        49.00   
75%            17.96            14.00                     137.00        74.00   
max            24.00            19.00                     179.00        99.00   

       AchievementsUnlocked  InGamePurchases  
count              40034.00          40034.0  
mean                  24.53              0.2  
std                   14.43              0.4  
min                    0.00              0.0  
25%                   12.00        

## 1. First attempt: 6 variables

K-means is distance-based, so the variables are standardised first (mean 0, standard deviation 1).
For each k we print the inertia (compactness, always decreases) and the silhouette score
(between -1 and 1, higher means better separated clusters). It is computed on a 10,000-player sample to stay fast.

In [2]:
X6 = StandardScaler().fit_transform(df[all_vars].dropna())

for k in range(2, 8):
    model = KMeans(n_clusters=k, n_init=10, random_state=42).fit(X6)
    sil = silhouette_score(X6, model.labels_, sample_size=10000, random_state=42)
    print(k, round(model.inertia_), round(sil, 3))

2 200165 0.237


3 176018 0.16


4 161546 0.144


5 149694 0.144


6 141010 0.142


7 133280 0.143


In [3]:
# Stability: same model, two different random seeds. ARI = 1 means the same groups, near 0 means unrelated groups.
m1 = KMeans(n_clusters=5, n_init=10, random_state=42).fit(X6)
m2 = KMeans(n_clusters=5, n_init=10, random_state=7).fit(X6)
print("ARI, 5 clusters (seed 42 vs 7):", round(adjusted_rand_score(m1.labels_, m2.labels_), 3))

ARI, 5 clusters (seed 42 vs 7): 0.463


**Reading.** The silhouette is low whatever k (0.24 at best, about 0.14 from k = 4), and the 5 clusters are
not stable across seeds (ARI about 0.46). This is a sign of weak structure.

## 2. Is there any structure? Real data vs shuffled data

We keep 4 variables that describe how often, how long, how far and whether the player pays
(`PlayTimeHours` and `AchievementsUnlocked` are dropped to avoid counting overall activity several times).
Then we shuffle each column independently: this destroys any link between variables, so the shuffled data
has no structure by construction. If the real silhouette is not clearly above the shuffled one, K-means
finds nothing that noise would not give.

In [4]:
vars4 = ["SessionsPerWeek", "AvgSessionDurationMinutes", "PlayerLevel", "InGamePurchases"]
X4_raw = df[vars4].dropna()
X4 = StandardScaler().fit_transform(X4_raw)

rng = np.random.default_rng(0)
X4_shuffled = np.column_stack([rng.permutation(X4[:, i]) for i in range(X4.shape[1])])

print("k | silhouette real | silhouette shuffled | ARI (seed 42 vs 7)")
for k in range(2, 7):
    real = KMeans(n_clusters=k, n_init=10, random_state=42).fit(X4)
    other = KMeans(n_clusters=k, n_init=10, random_state=7).fit(X4)
    noise = KMeans(n_clusters=k, n_init=10, random_state=42).fit(X4_shuffled)
    sil_real = silhouette_score(X4, real.labels_, sample_size=10000, random_state=42)
    sil_noise = silhouette_score(X4_shuffled, noise.labels_, sample_size=10000, random_state=42)
    ari = adjusted_rand_score(real.labels_, other.labels_)
    print(k, round(sil_real, 3), round(sil_noise, 3), round(ari, 3))

k | silhouette real | silhouette shuffled | ARI (seed 42 vs 7)


2 0.338 0.338 1.0


3 0.255 0.256 1.0


4 0.247 0.245 0.399


5 0.258 0.262 0.513


6 0.255 0.26 0.588


**Reading.** Real and shuffled silhouettes are almost identical: the clusters come from the shape of each
variable taken alone (one binary variable, the others spread evenly), not from relationships between variables.
k = 2 is just `InGamePurchases` (payers vs non-payers). k = 2 and k = 3 are stable across seeds (ARI = 1);
from k = 4 the clusters change with the seed (ARI 0.4 to 0.6).
We keep **k = 3**: payers, plus the non-payers split by how often they play. This is a descriptive
choice (readable and stable), not a statistical discovery.

## 3. Final model: 3 segments

In [5]:
final = KMeans(n_clusters=3, n_init=10, random_state=42).fit(X4)

seg = df.loc[X4_raw.index].copy()
seg["cluster"] = final.labels_

# Name the clusters from their averages: payers first, then the non-payers ranked by sessions per week
means = seg.groupby("cluster")[vars4].mean()
payers = means["InGamePurchases"].idxmax()
non_payers = means.drop(index=payers)["SessionsPerWeek"]
names = {payers: "Payers",
         non_payers.idxmax(): "Highly active non-payers",
         non_payers.idxmin(): "Low-activity non-payers"}
seg["segment"] = seg["cluster"].map(names)

print(seg.groupby("segment")[vars4].mean().round(2))
print(seg["segment"].value_counts(normalize=True).round(3))
# Cross-check with the engagement label that comes with the dataset (it was not used by the model)
print(pd.crosstab(seg["segment"], seg["EngagementLevel"], normalize="index").round(2))

                          SessionsPerWeek  AvgSessionDurationMinutes  \
segment                                                                
Highly active non-payers            14.52                      94.75   
Low-activity non-payers              4.51                      94.98   
Payers                               9.53                      94.49   

                          PlayerLevel  InGamePurchases  
segment                                                 
Highly active non-payers        49.64              0.0  
Low-activity non-payers         49.48              0.0  
Payers                          50.03              1.0  
segment
Low-activity non-payers     0.404
Highly active non-payers    0.395
Payers                      0.201
Name: proportion, dtype: float64
EngagementLevel           High   Low  Medium
segment                                     
Highly active non-payers  0.47  0.09    0.45
Low-activity non-payers   0.05  0.43    0.52
Payers                    0.27 

## Conclusion

**Question.** Are there distinct player profiles in these data?

**Method.** K-means on 4 standardised variables; k chosen from readability, after comparing with shuffled
data and checking stability across seeds.

**Result.** Correlations are close to zero and the silhouette is no better than on shuffled data: the structure
is weak. Three segments are kept for description:

| Segment | Share | Profile | Hypothesis (not tested) |
|---|---|---|---|
| Highly active non-payers | about 39% | many sessions, never paid | convert with an entry-level offer |
| Low-activity non-payers | about 40% | few sessions, never paid | re-engage with reminders and events |
| Payers | about 20% | paid at least once | retain with loyalty perks |

**Limits.** The data look simulated (uniform distributions, independent variables). The segments split payment
and session frequency; they do not describe real populations. The dataset's own engagement label follows
sessions per week (High: 47% of highly active non-payers vs 5% of low-activity ones), which is expected
and not a new finding.